# Data transformation methods

Axis 1: What boundary does it split at? <br>
Axis 2: How is chunk size measured?<br>

1. RecursiveCharTextSplitter -priority order: ["\n\n", "\n", " ", ""]. It tries to split on paragraphs first; if a resulting chunk is still too big, it recurses into that chunk and tries the next separator (sentences → words → characters)
2. CharTextSplitter  - Splits on a single separator (default "\n\n")
3. TokenTextSplitter
4. SentenceTransformerTokenTextSPlitter
5. NLTKTextSplitter
6. SpacyTextSplitter - More accurate for tricky sentence boundaries (abbreviations, etc.) but heavier dependency.
<br> Structure-aware splitters (format-specific)- These understand the format of the document, not just raw text, so they split along meaningful structural boundaries. <br>
7. MarkdownHeaderTextSplitter
8. MarkdownTextSplitter
<br> HTML <br>
9. HTMLHeaderTextSplitter
10. HTMLSectionSplitter
11. HTMLSectionSplitter
12. HTMLSemanticPreservingSplitter
13. RecursiveCharacterTextSplitter.from_language(Language.PYTHON)
14. RecursiveJsonSplitter

<br>
<b>chunk_overlap:</b> exists to solve one specific problem: important information sitting right at a chunk boundary getting cut in half and losing context.
<br>

Trade-off to be aware of, Overlap isn't free:<br>

More overlap → more redundant text stored → more embeddings computed → higher storage/<br>compute cost and slightly muddier retrieval (near-duplicate chunks competing for the same query)<br>
Too little/no overlap → risk of exactly the boundary-cutting problem above

<br>
Semantic chunking (SemanticChunker in langchain_experimental) — instead of fixed character/token rules, it splits where embedding similarity between sentences drops, so each chunk is more topically coherent. This is the more "advanced" technique that sometimes comes up in interviews as the answer to "how would you improve chunking quality."

In [ ]:


from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter,TokenTextSplitter
from langchain_community.document_loaders import TextLoader



That warning is the tell: CharacterTextSplitter only splits on one separator (default "\n\n"), and unlike RecursiveCharacterTextSplitter, it doesn't fall back to smaller separators when a chunk is still too big — it just accepts the oversized chunk and warns you.

Here's what happened:

Your speech has 7 paragraphs separated by \n\n.
CharacterTextSplitter splits only at those \n\n breaks — nothing else.
Each resulting paragraph is checked against chunk_size=100. Every single paragraph is way over 100 chars (347 to 982 chars).
Since there's no second separator to retry with, it just keeps each paragraph whole and logs "Created a chunk of size X, which is longer than the specified 100" for each one that overflows.
chunk_overlap=20 never even comes into play here, because overlap only applies between chunks that were actually split apart — these paragraphs were never split, just passed through as-is.

In [ ]:
#load data 
loader = TextLoader("speech.txt")
text_doc = loader.load()

#spliiter - #CharTextSplitter
splitter = CharacterTextSplitter(chunk_size = 500, chunk_overlap = 20)
print("****CharSplitter :split_text*******")
charSplit = splitter.split_text(text_doc[0].page_content)
print(len(charSplit))
print(charSplit[0])

print("****CharSplitter :split_doc*******")
docSplit = splitter.split_documents(text_doc)
print(len(docSplit))
print(docSplit)





RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20) — it'll drop down to splitting on \n, then spaces, until pieces are under 100 chars. CharacterTextSplitter is really only useful when your separator choice already produces reasonably-sized chunks (e.g., splitting well-formatted docs on \n\n where paragraphs are naturally short).

In [ ]:
print("****RecursiveCharSplitter :split_text*******")
seperator = ["\n\n", "\n", " ", ""]
splitter = RecursiveCharacterTextSplitter(separators=seperator,chunk_size = 100, chunk_overlap = 20)
textSplit = splitter.split_text(text_doc[0].page_content)
print(len(textSplit))
print(textSplit[0])
docSplit = splitter.split_documents(text_doc)
print(len(docSplit))
print(type(docSplit[0]))
print(docSplit[0])
print(docSplit[1])

TokenTextSplitter splits based on model tokens, not characters — that's the key difference from everything you've used so far.
How it works internally:

1. It uses tiktoken (OpenAI's tokenizer) to encode your whole text into a list of token IDs.
2. It slices that token list into windows of chunk_size tokens, with chunk_overlap tokens repeated between consecutive windows.
3. Each token window gets decoded back into a text string — that decoded string becomes your chunk.

Why TokenTextSplitter isn't the go-to despite being "more accurate" for tokens:

1. Speed — tokenizing with tiktoken on every document during ingestion adds real overhead, especially over large corpora. Character splitting is just string slicing — much faster.
2. It ignores text structure completely. Since it only counts tokens and cuts, it can slice a chunk mid-sentence or mid-word. For retrieval quality, semantically coherent chunks (whole sentences/paragraphs) usually embed and retrieve better than a chunk that starts mid-thought.
3. The character-to-token ratio is stable enough for English (~4 chars/token) that RecursiveCharacterTextSplitter(chunk_size=500) gets you into roughly the token range you want anyway, without the tokenization overhead.
4. Model-agnostic — TokenTextSplitter defaults to counting tokens the way tiktoken/OpenAI models do. If you switch to a different model family (Llama, Claude, etc.) with a different tokenizer, your "100 tokens" no longer means the same thing for that model. Character counts don't have this problem.<br>
<br>
Where TokenTextSplitter (or token-aware splitting) actually does get used:

1. When you're right up against a hard token ceiling — e.g., an embedding model with a strict max input (like text-embedding-3-small's 8191-token cap) and your documents have unusually dense/non-English text where character estimates would be unreliable.
2. Cost-sensitive pipelines at scale, where you want tight, predictable control over exact token spend per chunk (since embedding and LLM API billing is per-token).
3. Some teams use a hybrid approach: split with RecursiveCharacterTextSplitter for structure-awareness, but pass a token-counting function (via tiktoken) as the length_function parameter, so chunk boundaries still respect paragraphs/sentences while the size check is token-based instead of character-based. That's actually a common middle ground. <br>

<br>
Bottom line for your course/portfolio work: RecursiveCharacterTextSplitter is what you'll see in the vast majority of LangChain RAG tutorials and production code — Krish Naik's course is likely emphasizing it for that reason. Know TokenTextSplitter exists and when it's the right tool, but don't expect to reach for it often.







In [ ]:
#Token based splitter
splitter = TokenTextSplitter(chunk_size = 100, chunk_overlap=10)
textSplit = splitter.split_text(text_doc[0].page_content)
print(len(textSplit))
print(textSplit)


# HTML Header split

In [ ]:
from langchain_text_splitters import HTMLHeaderTextSplitter

header = [
    ("h1","header1"),
    ("h2","header2"),
    ("h3","header3")
]
splitter = HTMLHeaderTextSplitter(headers_to_split_on=header)
htmlText = splitter.split_text_from_file("example.html")
print(type(htmlText),len(htmlText))
print(htmlText)
header1_text = [content.page_content for content in htmlText if content.metadata.get("header1") and not content.metadata.get("header2") and not content.metadata.get("header3")]
print(len(header1_text))
print(header1_text)

# RecusiveJSON Splitter

Rule of thumb: use JSONLoader when you want jq to pick out specific records as separate Documents (headed toward RecursiveCharacterTextSplitter or straight to embedding). Use RecursiveJsonSplitter when you want to preserve one JSON's nested structure while breaking it into size-bounded pieces — feed it raw dicts via json.load(), not Document.page_content strings from a loader.

Good example to look at closely — and it exposes an important limitation of RecursiveJsonSplitter: max_chunk_size is a target, not a hard cap, similar to what you saw with CharacterTextSplitter overshooting earlier.

What's happening:

Chunk index 4 (departments) is massive — way more than 10 characters worth of content — despite max_chunk_size=10. Here's why: RecursiveJsonSplitter only splits at key boundaries within nested dicts. It walks the JSON tree and tries to break pieces apart at natural dict/list boundaries so it doesn't exceed max_chunk_size. But when it hits a list of large objects (like departments, which is a list of department dicts, each containing nested teams and projects), it doesn't know how to split inside a list element cleanly the same way it splits dict keys — so if a single list item is bigger than max_chunk_size, it just keeps it whole rather than breaking it further.

In [2]:
from langchain_text_splitters import RecursiveJsonSplitter
import json
# from langchain_community.document_loaders import JSONLoader

with open(file="sample.json") as f:
    jsondata = json.load(f)

splitter = RecursiveJsonSplitter(max_chunk_size = 10)
jsonSplit = splitter.split_json(json_data=jsondata)
print(len(jsonSplit))
print(jsonSplit[0].keys())
print(jsonSplit[1].keys(),jsonSplit[1].values())
print(jsonSplit[2].keys(),jsonSplit[2].values())
print(jsonSplit[3].keys(),jsonSplit[3].values())
print(jsonSplit[4].keys(),jsonSplit[4].values())
print(jsonSplit[5].keys())

7
dict_keys(['company', 'founded'])
dict_keys(['headquarters']) dict_values([{'city': 'Bengaluru', 'state': 'Karnataka'}])
dict_keys(['headquarters']) dict_values([{'country': 'India', 'address': {'street': '12 MG Road'}}])
dict_keys(['headquarters']) dict_values([{'address': {'zip': '560001', 'landmark': 'Near Trinity Metro Station'}}])
dict_keys(['departments']) dict_values([[{'name': 'Engineering', 'head': 'Ananya Rao', 'teams': [{'team_name': 'Backend', 'members': 12, 'tech_stack': ['Java', 'Spring Boot', 'PostgreSQL'], 'current_projects': [{'project_id': 'ENG-101', 'title': 'Payment Gateway Revamp', 'status': 'in_progress', 'priority': 'high'}, {'project_id': 'ENG-102', 'title': 'API Rate Limiter', 'status': 'completed', 'priority': 'medium'}]}, {'team_name': 'AI/ML', 'members': 8, 'tech_stack': ['Python', 'PyTorch', 'LangChain', 'HuggingFace'], 'current_projects': [{'project_id': 'ML-201', 'title': 'RAG-based Customer Support Bot', 'status': 'in_progress', 'priority': 'high', 'de

# Interview questions



Coding <br>

Problem 1 <br>

Split a document using:<br>

chunk_size = 500<br>
chunk_overlap = 50<br>

and inspect the resulting chunks.<br>



In [ ]:
#problem 1
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

#load document
loader = PyPDFLoader(file_path="/Users/sanvijanvi/Documents/SJ_HomeWork/attention.pdf")
pdfDoc = loader.load()

#split the document
seperators = ["\n\n","\n"," ",""]
spliiter=RecursiveCharacterTextSplitter(separators=seperators,chunk_size = 500, chunk_overlap = 50)
recursiveSplitDocs = spliiter.split_documents(pdfDoc)

print(len(recursiveSplitDocs),type(recursiveSplitDocs),type(recursiveSplitDocs[0]),recursiveSplitDocs[0])
# for splitDoc in pdfDoc_split:
#     print(splitDoc)


Problem 2<br>

Compare:<br>

CharacterTextSplitter<br>
RecursiveCharacterTextSplitter<br>
TokenTextSplitter<br>

on the same document.<br>



In [ ]:
#problem 2 
from langchain_text_splitters import CharacterTextSplitter, TokenTextSplitter

splitter =  CharacterTextSplitter(chunk_size = 500, chunk_overlap = 50)
charSplitDocs = splitter.split_documents(pdfDoc)

splitter =  TokenTextSplitter(chunk_size = 500, chunk_overlap = 50)
tokenSplitDocs = splitter.split_documents(pdfDoc)

#comapre all thre splitters
print(len(recursiveSplitDocs), len(charSplitDocs),len(tokenSplitDocs)) #93 15 28
print(len(recursiveSplitDocs[0].page_content), len(charSplitDocs[0].page_content),len(tokenSplitDocs[0].page_content))#492 2859 1916


Problem 3 ⭐<br>

Create a program that prints:<br>

Chunk ID<br>
Characters<br>
Tokens<br>
Preview<br>

for every chunk.<br>



In [ ]:
#problem 3
for i, doc in enumerate(recursiveSplitDocs):
    print("chunk id:",i)
    # print("characters:",doc.page_content)
    


Problem 4 ⭐<br>

Experiment with:<br>

chunk_size = 200<br>
500<br>
1000<br>

and compare retrieval results.<br>



In [ ]:
#problem 4
chunk_sizes = [200,500,1000]

for chnSize in chunk_sizes:
    pdfSplitter = RecursiveCharacterTextSplitter(separators=seperator, chunk_size = chnSize,chunk_overlap = 50)
    pdfChunks = pdfSplitter.split_documents(pdfDoc)
    print(f"Chunk size : {chnSize}, pdf chunks:{len(pdfChunks)},length of each chunk:{len(pdfChunks[0].page_content)}")




Problem 5 Advanced<br>

Create a chunking strategy for a technical PDF where:<br>
<br>
Chapter<br>
   ↓<br>
Section<br>
      ↓<br>
Paragraph<br>

structure should be preserved as much as possible.<br>

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/Users/sanvijanvi/Documents/SJ_HomeWork/distributed_caching_practice.pdf")
pdfDocs = loader.load()

splitter = RecursiveCharacterTextSplitter(separators=seperator,chunk_size = 500, chunk_overlap = 100)
pdfChunks = splitter.split_documents(pdfDocs)

for chunk in pdfChunks:
    print("********")
    print(chunk.page_content)